In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, zscore
from scipy.special import logit, expit
from statsmodels.stats.multitest import multipletests
from ALLCools.mcds import MCDS 

# Define paths
mcds_path = '/ceph/MethDev/pbio/hazel/gene_mcds/gene'
rna_path = '/ceph/MethDev/pbio/hazel/count_Arab13_with_TEs.csv'
meta_path = '/ceph/MethDev/pbio/hazel/merged_cluster_assignments.csv'

# Load datasets
mcds = MCDS.open(mcds_paths=mcds_path, var_dim='gene')
rna = pd.read_csv(rna_path).T
meta = pd.read_csv(meta_path).set_index('cell_id')

print(f"Loaded RNA shape: {rna.shape}")
print(f"Loaded Meta shape: {meta.shape}")

Loaded RNA shape: (6071, 64507)
Loaded Meta shape: (5908, 1)


In [2]:
import pandas as pd
import re

gff_path = '/gale/raidix/rdx-7/jwalker/Kay_Qu/c1038_genes_overlapping_c2952_dmw_chunks.gff'

# 1. Read the GFF (Skipping comment lines starting with '#')
# GFFs are tab-separated and have 9 standard columns
gff_cols = ['seqid', 'source', 'type', 'start', 'end', 'score', 'strand', 'phase', 'attributes']
df_gff = pd.read_csv(gff_path, sep='\t', comment='#', header=None, names=gff_cols)

# 2. Filter for 'gene' features only (Column 3)
genes_only = df_gff[df_gff['type'] == 'gene'].copy()

# 3. Extract the ID from the 'attributes' column
# Usually looks like: "ID=gene:AT1G01010;Name=..." or "ID=AT1G01010"
def extract_id(attr_string):
    # This regex looks for 'ID=' followed by characters until a semicolon or end of string
    match = re.search(r'ID=([^;]+)', attr_string)
    if match:
        gene_id = match.group(1)
        # Clean up common prefixes like 'gene:' if they exist
        return gene_id.replace('gene:', '')
    return None

# Apply the extraction
new_gene_list = genes_only['attributes'].apply(extract_id).dropna().unique().tolist()

# 4. Update your pipeline variable
hazel_genes = pd.Index(new_gene_list)

print(f"✅ Loaded {len(hazel_genes)} unique Gene IDs from the GFF file.")
print(f"Sample IDs: {new_gene_list[:5]}")

✅ Loaded 1038 unique Gene IDs from the GFF file.
Sample IDs: ['AT1G01300', 'AT1G01770', 'AT1G01780', 'AT1G01830', 'AT1G01950']


In [3]:
# ==========================================
# CELL 3: BUILD UNCOLLAPSED RNA MATRIX
# ==========================================
# 1. Define the mapping and desired order
cluster_mapping = {
    3: '3 Early M', 0: '0 Mid M', 7: '7 Expanding M', 1: '1 Late M', 10: '10 Senescent M',
    5: '5 Early E', 4: '4 Abaxial E', 6: '6 Adaxial E', 13: '13 Expanding E', 8: '8 Senescent E',
    14: '14 Guard cell', 2: '2 Vasculature', 11: '11 PPP', 9: '9 Phloem', 
    16: '16 Myrosinase', 12: '12 S phase', 15: '15 G2M phase'
}
# Support string keys in case index types vary
cluster_mapping.update({str(k): v for k, v in cluster_mapping.items()})

cluster_order = [
    '3 Early M', '0 Mid M', '7 Expanding M', '1 Late M', '10 Senescent M',
    '5 Early E', '13 Expanding E', '4 Abaxial E', '6 Adaxial E', '8 Senescent E',
    '14 Guard cell', '2 Vasculature', '11 PPP', '9 Phloem', 
    '16 Myrosinase', '12 S phase', '15 G2M phase'
]

# 2. Align cells
common_cells = rna.index.intersection(meta.index)
rna_aligned = rna.loc[common_cells]

# 3. Normalize to CPM and Log1p transform
cell_totals = rna_aligned.sum(axis=1)
rna_cpm = rna_aligned.div(cell_totals, axis=0) * 1e6
rna_log1p = np.log1p(rna_cpm)

# 4. Map the fine-grained clusters instead of 'merged_cluster'
# Assuming the single column in your meta file holds the cluster ID (0-16)
meta_col_name = meta.columns[0]
fine_clusters = meta.loc[common_cells, meta_col_name].map(cluster_mapping)
rna_log1p['fine_cluster'] = fine_clusters

# 5. Aggregate by the new fine-grained clusters
uncollapsed_rna_matrix = rna_log1p.groupby('fine_cluster').mean().T
uncollapsed_rna_matrix = uncollapsed_rna_matrix[cluster_order] # Force correct order

print(f"Uncollapsed RNA matrix shape: {uncollapsed_rna_matrix.shape}")


# ==========================================
# CELL 4: BUILD UNCOLLAPSED METHYLATION MATRIX
# ==========================================
# 1. Get raw counts (c = methylated, m = total coverage)
mCGN_mc = mcds.sel(mc_type='CGN', count_type='mc').gene_da.to_pandas()
mCGN_cov = mcds.sel(mc_type='CGN', count_type='cov').gene_da.to_pandas()

# Map the fine-grained clusters
mCGN_mc['fine_cluster'] = meta[meta_col_name].map(cluster_mapping)
mCGN_cov['fine_cluster'] = meta[meta_col_name].map(cluster_mapping)

# Aggregate counts by fine cluster
cluster_c = mCGN_mc.groupby('fine_cluster').sum().T.reindex(columns=cluster_order)
uncollapsed_cluster_m = mCGN_cov.groupby('fine_cluster').sum().T.reindex(columns=cluster_order)

# 2. Fit Global Offsets (deltas)
global_c = cluster_c.sum(axis=0)
global_m = uncollapsed_cluster_m.sum(axis=0)
global_p = global_c / global_m

d = logit(np.clip(global_p, 1e-6, 1-1e-6))
deltas = d - np.average(d, weights=global_m)

# 3. Calculate Expected Methylation (p0)
gene_c = cluster_c.sum(axis=1)
gene_m = uncollapsed_cluster_m.sum(axis=1)

valid_genes_mask = gene_m >= 50 
cluster_c = cluster_c[valid_genes_mask]
cluster_m_filt = uncollapsed_cluster_m[valid_genes_mask]
gene_c = gene_c[valid_genes_mask]
gene_m = gene_m[valid_genes_mask]

pbar = np.clip(gene_c / gene_m, 1e-6, 1-1e-6)
logit_pbar = logit(pbar).values[:, None]
expected_logit = logit_pbar + deltas.values[None, :]
p0_df = pd.DataFrame(expit(expected_logit), index=cluster_c.index, columns=cluster_c.columns)

# 4. Extract Adjusted Matrix
obs_p = cluster_c / cluster_m_filt.replace(0, np.nan)
obs_p = obs_p.fillna(0) 

uncollapsed_adj_meth_matrix = obs_p - p0_df

print(f"Uncollapsed Meth Residuals matrix shape: {uncollapsed_adj_meth_matrix.shape}")

Uncollapsed RNA matrix shape: (64507, 17)
Uncollapsed Meth Residuals matrix shape: (32357, 17)


In [4]:
uncollapsed_adj_meth_matrix

fine_cluster,3 Early M,0 Mid M,7 Expanding M,1 Late M,10 Senescent M,5 Early E,13 Expanding E,4 Abaxial E,6 Adaxial E,8 Senescent E,14 Guard cell,2 Vasculature,11 PPP,9 Phloem,16 Myrosinase,12 S phase,15 G2M phase
gene,,,,,,,,,,,,,,,,,
AT1G01010,-8.262579e-04,-2.953897e-03,-0.001566,0.001160,-4.767285e-03,0.005403,-0.002880,0.002184,-1.200140e-03,8.535093e-03,0.000439,0.000922,-0.011367,2.767280e-03,-7.534387e-03,6.203095e-03,0.014476
AT1G01020,-6.900968e-03,7.373509e-03,-0.007073,0.003458,-3.806587e-03,0.030758,-0.013520,-0.008512,7.918238e-03,-1.436805e-02,-0.000611,-0.011837,0.004967,-3.758803e-02,3.132104e-02,-4.259474e-03,-0.034999
AT1G03987,-1.187067e-02,-3.158764e-03,0.033425,-0.013704,-9.275444e-03,-0.013091,-0.013257,0.008021,-1.270397e-02,1.304301e-02,-0.014784,-0.001659,-0.013437,1.914312e-02,-1.178785e-02,8.751022e-02,0.110927
AT1G01030,6.833176e-03,-6.770089e-06,-0.001288,-0.001300,1.204374e-03,-0.000723,-0.000913,-0.000345,2.952175e-03,2.069413e-04,-0.005619,-0.001510,-0.004692,-5.473740e-04,1.951287e-03,2.900375e-04,-0.007480
AT1G01040,-1.461289e-02,-8.554127e-03,0.020724,0.025925,-6.590146e-02,0.012230,-0.052523,0.002149,-1.924413e-02,-5.517670e-02,0.059192,0.039956,-0.015006,-3.030103e-02,1.937139e-02,-1.400378e-02,0.010152
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ATCG01230,-9.310287e-07,-9.699846e-07,-0.000001,-0.000001,-7.255771e-07,-0.000001,-0.000001,-0.000001,-9.972267e-07,-7.578702e-07,-0.000001,-0.000001,-0.000001,-8.746546e-07,-9.244560e-07,-9.802004e-07,-0.000001
ATCG01240,-9.310287e-07,-9.699846e-07,-0.000001,-0.000001,-7.255771e-07,-0.000001,-0.000001,-0.000001,-9.972267e-07,-7.578702e-07,-0.000001,-0.000001,-0.000001,-8.746546e-07,-9.244560e-07,-9.802004e-07,-0.000001
ATCG01250,-1.775721e-02,-1.848647e-02,0.644319,-0.020481,-1.389313e-02,-0.019571,-0.019817,-0.020501,-1.899580e-02,-1.450250e-02,-0.022083,-0.020272,-0.020085,-1.669996e-02,-1.763406e-02,-1.867753e-02,-0.021029


In [5]:
# ==========================================
# REFACTORED COMBINATORIAL HEATMAP SCRIPT
# ==========================================
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as hc
import scipy.spatial as sp
import statsmodels.api as sm
from scipy.stats import zscore
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import LinearSegmentedColormap

teal_rust_cmap = LinearSegmentedColormap.from_list("Teal_Rust", ["#FF8C00", "white", "#004d4d"])
rna_cmap = LinearSegmentedColormap.from_list("Grey_Green", ["white", "forestgreen"])

# Re-assign variables for clean reading
rna_df = uncollapsed_rna_matrix
meth_df = uncollapsed_adj_meth_matrix
weights_df = uncollapsed_cluster_m

# 1. Define the 3 Cluster Dimensions
meso_epi_cols = [c for c in cluster_order if c.endswith('M') or c.endswith('E')]
non_meso_epi_cols = [c for c in cluster_order if c not in meso_epi_cols]

cluster_dims = {
    'all_clusters': cluster_order,
    'meso_epi': meso_epi_cols,
    'non_meso_epi': non_meso_epi_cols
}

# 2. Define the 2 Gene Dimensions
all_metadata_genes = rna_df.index.intersection(meth_df.index)
gene_dims = {
    '30k_genes': all_metadata_genes,
    'hazel_genes': all_metadata_genes.intersection(hazel_genes)
}

# ==========================================
# HELPER FUNCTIONS
# ==========================================
def calculate_wls_stats(genes, subset_cols):
    """Calculates WLS specifically on the selected column subset."""
    valid_genes, p_vals = [], []
    for gene in genes:
        r_vals = rna_df.loc[gene, subset_cols].values
        m_vals = meth_df.loc[gene, subset_cols].values
        w_vals = weights_df.loc[gene, subset_cols].values
        
        if np.std(r_vals) == 0 or np.std(m_vals) == 0:
            continue
            
        try:
            mod = sm.WLS(zscore(r_vals), sm.add_constant(zscore(m_vals)), weights=w_vals).fit()
            valid_genes.append(gene)
            p_vals.append(mod.pvalues[1])  # index 1 is the meth coefficient
        except:
            continue

    stats_df = pd.DataFrame({'pval': p_vals}, index=valid_genes)
    if len(stats_df) > 0:
        stats_df['fdr'] = multipletests(stats_df['pval'], method='fdr_bh')[1]
    
    return stats_df

def get_clustered_data(genes, cols):
    """Z-scores and hierarchically clusters genes for the target columns."""
    z_meth = meth_df.loc[genes, cols].apply(zscore, axis=1).clip(-2.5, 2.5).fillna(0)
    z_rna = rna_df.loc[genes, cols].apply(zscore, axis=1).clip(-2.5, 2.5).fillna(0)
    
    if len(genes) > 1:
        linkage = hc.linkage(sp.distance.pdist(z_meth.values), method='ward')
        order = hc.leaves_list(linkage)
        return z_meth.iloc[order], z_rna.iloc[order]
    return z_meth, z_rna


def get_clustered_data_anchored(genes, plot_cols, anchor_cols):
    """
    Z-scores and clusters genes based ONLY on the anchor_cols,
    but outputs the data for all plot_cols to show expansion.
    """
    # 1. Calculate Mean and Std ONLY on the anchor (subset) columns
    # Using ddof=0 to perfectly match scipy.stats.zscore behavior
    meth_mean = meth_df.loc[genes, anchor_cols].mean(axis=1)
    meth_std = meth_df.loc[genes, anchor_cols].std(axis=1, ddof=0).replace(0, 1) 
    
    rna_mean = rna_df.loc[genes, anchor_cols].mean(axis=1)
    rna_std = rna_df.loc[genes, anchor_cols].std(axis=1, ddof=0).replace(0, 1)
    
    # 2. Apply this subset-derived Mean & Std to ALL requested plot columns
    z_meth_plot = meth_df.loc[genes, plot_cols].sub(meth_mean, axis=0).div(meth_std, axis=0).clip(-2.5, 2.5).fillna(0)
    z_rna_plot = rna_df.loc[genes, plot_cols].sub(rna_mean, axis=0).div(rna_std, axis=0).clip(-2.5, 2.5).fillna(0)
    
    # 3. Perform Hierarchical Clustering ONLY on the anchor columns
    z_meth_anchor = z_meth_plot[anchor_cols] 
    
    if len(genes) > 1:
        linkage = hc.linkage(sp.distance.pdist(z_meth_anchor.values), method='ward')
        order = hc.leaves_list(linkage)
        
        # Apply the subset-driven row order to the full dataset
        return z_meth_plot.iloc[order], z_rna_plot.iloc[order]
        
    return z_meth_plot, z_rna_plot

def plot_standalone_heatmap(genes, plot_cols, title, save_path, anchor_cols=None):
    """Generates a standalone 1x2 heatmap, with optional anchoring for Z-scores/clustering."""
    if len(genes) == 0:
        print(f"⚠️ Skipping plot '{title}' - no genes survived filters.")
        return
        
    # Check if we should use the anchored calculation or the standard one
    if anchor_cols is not None:
        m_plot, r_plot = get_clustered_data_anchored(genes, plot_cols, anchor_cols)
    else:
        m_plot, r_plot = get_clustered_data(genes, plot_cols)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # Meth Heatmap - Lock vmin and vmax!
    sns.heatmap(m_plot, cmap="coolwarm", center=0, vmin=-2.5, vmax=2.5, 
                ax=axes[0], cbar_kws={'label': 'Meth Z'}, yticklabels=False)
    axes[0].set_title(f"{title}\nMethylation Residuals (n={len(genes)})", fontsize=14, pad=10)
    
    # RNA Heatmap - Lock vmin and vmax!
    sns.heatmap(r_plot, cmap=teal_rust_cmap, vmin=-2.5, vmax=2.5, 
                ax=axes[1], cbar_kws={'label': 'RNA Z'}, yticklabels=False)
    axes[1].set_title(f"{title}\nRNA Expression (n={len(genes)})", fontsize=14, pad=10)

    # Dynamic Formatting & Dividers
    for ax in axes:
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=11)
        
        if len(plot_cols) == 17:
            ax.axvline(5, color='black', lw=2, linestyle=':')  
            ax.axvline(10, color='black', lw=2, linestyle=':')
        elif len(plot_cols) == 10:
            ax.axvline(5, color='black', lw=2, linestyle=':')

    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close() 
    print(f"✅ Saved: {save_path}")

# ==========================================
# EXECUTE COMBINATORIAL PIPELINE
# ==========================================

save_dir = '/ceph/MethDev/pbio/kay/figures/heatmaps_combinatorial/'

for c_name, c_cols in cluster_dims.items():
    for g_name, g_subset in gene_dims.items():
        
        # 1. Run WLS using ONLY the current cluster subset
        stats_df = calculate_wls_stats(g_subset, c_cols)
        
        if len(stats_df) == 0:
            print(f"⚠️ No valid WLS results for {c_name} x {g_name}. Skipping...")
            continue
            
        # 2. Extract Genes based on the 2 Filtering Logics
        filter_outputs = {
            'fdr_less_0.05': stats_df[stats_df['fdr'] < 0.05].index,
            'wls_top_300': stats_df.sort_values('fdr').head(300).index
        }
        
        # 3. Generate the Plots
        for f_name, filtered_genes in filter_outputs.items():
            
            base_filename = f"{c_name}_{g_name}_{f_name}"
            base_title = f"WLS on {c_name} | {g_name} | {f_name}"
            
# Condition A: Plot the genes using the specific cluster subset (No anchor needed)
            plot_standalone_heatmap(
                genes=filtered_genes, 
                plot_cols=c_cols, 
                title=f"{base_title}\n(Plotted on {c_name} columns)", 
                save_path=os.path.join(save_dir, f"{base_filename}_target_cols.png")
            )
            
            # Condition B: Expand to all 17 columns, but ANCHOR to the subset
            if c_name != 'all_clusters':
                plot_standalone_heatmap(
                    genes=filtered_genes, 
                    plot_cols=cluster_order, # The 17 columns to plot
                    title=f"{base_title}\n(Expanded to ALL 17 columns)", 
                    save_path=os.path.join(save_dir, f"{base_filename}_expanded.png"),
                    anchor_cols=c_cols       # Tell the function to anchor math to the subset!
                )

print("\n🎉 Pipeline complete! All combinatorial heatmaps generated.")

✅ Saved: ./figures/heatmaps_combinatorial/all_clusters_30k_genes_fdr_less_0.05_target_cols.png
✅ Saved: ./figures/heatmaps_combinatorial/all_clusters_30k_genes_wls_top_300_target_cols.png
✅ Saved: ./figures/heatmaps_combinatorial/all_clusters_hazel_genes_fdr_less_0.05_target_cols.png
✅ Saved: ./figures/heatmaps_combinatorial/all_clusters_hazel_genes_wls_top_300_target_cols.png
⚠️ Skipping plot 'WLS on meso_epi | 30k_genes | fdr_less_0.05
(Plotted on meso_epi columns)' - no genes survived filters.
⚠️ Skipping plot 'WLS on meso_epi | 30k_genes | fdr_less_0.05
(Expanded to ALL 17 columns)' - no genes survived filters.
✅ Saved: ./figures/heatmaps_combinatorial/meso_epi_30k_genes_wls_top_300_target_cols.png
✅ Saved: ./figures/heatmaps_combinatorial/meso_epi_30k_genes_wls_top_300_expanded.png
✅ Saved: ./figures/heatmaps_combinatorial/meso_epi_hazel_genes_fdr_less_0.05_target_cols.png
✅ Saved: ./figures/heatmaps_combinatorial/meso_epi_hazel_genes_fdr_less_0.05_expanded.png
✅ Saved: ./figures/

In [6]:
# 1. Get the specific genes for this test
# (Assuming you already have stats_df calculated for Meso+Epi and 30k genes)
test_genes = stats_df.sort_values('fdr').head(300).index

# 2. Get the row-ordered data for the NON-EXPANDED plot (Standard)
m_plot_standard, _ = get_clustered_data(test_genes, meso_epi_cols)
standard_order = m_plot_standard.index.tolist()

# 3. Get the row-ordered data for the EXPANDED plot (Anchored)
m_plot_anchored, _ = get_clustered_data_anchored(test_genes, cluster_order, anchor_cols=meso_epi_cols)
anchored_order = m_plot_anchored.index.tolist()

# 4. Compare the top 10
print("Top 10 Genes - Non-Expanded (Standard):")
print(standard_order[:10])

print("\nTop 10 Genes - Expanded (Anchored):")
print(anchored_order[:10])

print(f"\nAre the full lists EXACTLY identical? {standard_order == anchored_order}")

Top 10 Genes - Non-Expanded (Standard):
['AT2G07669', 'AT2G07599', 'AT2G07667', 'AT2G07798', 'AT2G07835', 'AT2G07795', 'AT2G07787', 'AT2G07806', 'AT2G07698', 'AT2G07827']

Top 10 Genes - Expanded (Anchored):
['AT2G07669', 'AT2G07599', 'AT2G07667', 'AT2G07798', 'AT2G07835', 'AT2G07795', 'AT2G07787', 'AT2G07806', 'AT2G07698', 'AT2G07827']

Are the full lists EXACTLY identical? True


In [7]:
# 1. Calculate WLS specifically for non-meso-epi columns and the 30k genes
print("Calculating WLS stats for non-meso-epi...")
stats_df_non_meso = calculate_wls_stats(all_metadata_genes, non_meso_epi_cols)

# 2. Extract the Top 300 genes based on lowest FDR
test_genes_non_meso = stats_df_non_meso.sort_values('fdr').head(300).index

# 3. Get the row-ordered data for the NON-EXPANDED plot (Standalone)
m_plot_standard, _ = get_clustered_data(test_genes_non_meso, non_meso_epi_cols)
standard_order = m_plot_standard.index.tolist()

# 4. Get the row-ordered data for the EXPANDED plot (Anchored to non-meso-epi)
m_plot_anchored, _ = get_clustered_data_anchored(test_genes_non_meso, cluster_order, anchor_cols=non_meso_epi_cols)
anchored_order = m_plot_anchored.index.tolist()

# 5. Compare the top 10 genes
print("\n--- RESULTS ---")
print("Top 10 Genes - Non-Expanded (Standard):")
print(standard_order[:10])

print("\nTop 10 Genes - Expanded (Anchored):")
print(anchored_order[:10])

# 6. Check the entire 300-gene list for perfect alignment
print(f"\nAre the full lists EXACTLY identical? {standard_order == anchored_order}")

Calculating WLS stats for non-meso-epi...

--- RESULTS ---
Top 10 Genes - Non-Expanded (Standard):
['AT2G37990', 'AT4G16650', 'AT5G41440', 'AT1G51250', 'AT5G15510', 'AT2G42000', 'AT5G49420', 'AT5G55180', 'AT2G25565', 'AT5G00985']

Top 10 Genes - Expanded (Anchored):
['AT2G37990', 'AT4G16650', 'AT5G41440', 'AT1G51250', 'AT5G15510', 'AT2G42000', 'AT5G49420', 'AT5G55180', 'AT2G25565', 'AT5G00985']

Are the full lists EXACTLY identical? True
